# 05 · Temporal design: which scheduled inspections will come in low

Notebooks 02–04 follow Chelsea: hide part of the city and score it. Dallas inspects every
multitenant property, so the question that matters operationally is different: at the
start of a fiscal year, which of the properties due for inspection will score in the
bottom 10%? That ordering could set visit priority or re-inspection spacing.

**Unit.** Property × fiscal year: the property's first scored inspection that year
(5,377 units, FY2021-22 to FY2024-25; FY2020-21 serves as history only).

**Target.** That inspection scored ≤ 86 (bottom 10% across all inspections).

**Information.** Everything is as of the start of the fiscal year (1 October):

- the property's own inspection history from earlier fiscal years: number of inspections,
  last, minimum and mean score, prior bottom-10 count, score trend, time since the last
  inspection, prior inspection flags;
- owner groups' and the 10 nearest properties' bottom-10 record in earlier years
  (mailing-address sponsor and CoStar true owner; the property itself excluded);
- eviction filings (own parcel plus adjacent same-owner parcels) and resident 311 Code
  Compliance requests within 100 m, in the 12 and 36 months before the year starts;
- the static features from notebooks 02–04 that predate the window (tract ACS, DCAD
  values and size, DCAD 2016–2021 history, CoStar class and stars, spatial context).

**Evaluation.** Rolling origin. Train on FY2021-22 and FY2022-23, test on FY2023-24; train
on FY2021-22 to FY2023-24, test on FY2024-25. Results by test year and pooled (the
ensemble ranks within each year before pooling).

**Bars to clear.** Ranking by last score, which approximates the risk-based schedule
Dallas already runs, and ranking by whether the property was ever in the bottom 10%.

## Findings

Run 2026-10-01. Outputs are cleared before commit; the numbers below are from that run.

Pooled over FY2023-24 and FY2024-25 (2,923 test units, prevalence 0.110).

| Ranker | PR-AUC | Share of bottom-10 in top 20% | ROC-AUC |
|---|---|---|---|
| Random | 0.110 | 20% | 0.50 |
| Poverty only | 0.139 | 27% | 0.59 |
| Ever bottom-10 before | 0.132 | 26% | 0.55 |
| Last score (lower first) | 0.171 | 37% | 0.67 |
| Static features only (02 set), ensemble | 0.182 | 37% | 0.66 |
| Own history only, ensemble | 0.200 | 39% | 0.67 |
| **Static + own history, ensemble** | **0.230** | **40%** | **0.71** |
| + owner and neighbour record | 0.228 | 41% | 0.71 |
| + evictions and 311 before the year | 0.230 | 40% | 0.71 |
| + static supplemental (everything) | 0.210 | 40% | 0.71 |

- **The best temporal model** (static features plus own inspection history) reaches 0.23
  against a 0.11 base rate, about 2.1×. By year it scores 0.18 in FY2023-24 and 0.28 in
  FY2024-25. Last score alone gets 0.17, so the model adds 6 points over the rule closest to
  Dallas's risk-based schedule.
- **Adding owner, neighbour, eviction and 311 features doesn't help.** With about
  2,500 training units, more features add noise.
- **Properties with no inspection history** (363 test units, prevalence 0.08) are where the
  model adds most. XGBoost reaches 0.24 there, where the last-score rule has nothing to go
  on. Among properties with history, the model (0.18) and last score (0.18) tie.
- **Year-to-year persistence is weak,** which caps this design. Notebook 06 looks at why.
- **Leak check:** shuffled targets score at prevalence (0.10–0.11 against 0.11).

In [ ]:
from __future__ import annotations

import warnings

import numpy as np
import pandas as pd
from sklearn.inspection import permutation_importance

import features as f
import holdout as h
import temporal as t

warnings.filterwarnings("ignore", message="Skipping features without any observed values")
pd.set_option("display.float_format", "{:.3f}".format)

base = h.load_base()
COSTAR, costar_owner = f.costar(base)
CONTEXT, HIST, TAX = f.spatial_context(base), f.dcad_history(base), f.tax_delinquency(base)
base = pd.concat([base, CONTEXT, HIST, TAX, COSTAR], axis=1)

insp = t.load_inspections()
units = t.build_units(insp, base, owner_keys={"costar_owner": costar_owner})
units = t.add_prior_activity(units, base, f.eviction_parcel_sets(base))
print(units.groupby("fiscal_year").agg(
    units=("target", "size"), bottom10=("target", "mean"), with_history=("has_history", "mean")
).round(3).to_string())

## Configurations

In [ ]:
STATIC = h.BASELINE_FEATURES
STATIC_PLUS = (
    STATIC + list(CONTEXT.columns)
    + [c for c in HIST.columns if c != "sold_during_window"]
    + ["delinquent_since_pre2021", "costar_class", "costar_stars", "costar_affordable",
       "costar_unit_sf", "costar_years_since_renovated"]
)
HISTORY = t.HISTORY_COLS
PRIOR_CONTEXT = t.CONTEXT_COLS + ["costar_owner_prior_b10_rate", "costar_owner_prior_n"]
ACTIVITY = t.ACTIVITY_COLS

CONFIGS = {
    "static (02 baseline)": STATIC,
    "own history only": HISTORY,
    "static + own history": STATIC + HISTORY,
    "+ owner and neighbour record": STATIC + HISTORY + PRIOR_CONTEXT,
    "+ evictions and 311 before the year": STATIC + HISTORY + PRIOR_CONTEXT + ACTIVITY,
    "+ static supplemental (everything)": STATIC_PLUS + HISTORY + PRIOR_CONTEXT + ACTIVITY,
}
BASELINES = {
    "Last score (lower first)": -units.last_score.fillna(units.last_score.median()),
    "Ever bottom-10 before": units.any_prior_b10.fillna(0) + 0.01 * units.n_prior_b10.fillna(0),
    "Poverty only": units.pct_poverty,
}
results = pd.concat(
    [t.run(units, feats, label=name, baselines=BASELINES) for name, feats in CONFIGS.items()],
    ignore_index=True,
)

In [ ]:
def wide(metric: str) -> pd.DataFrame:
    return results.pivot_table(index=["label", "ranker"], columns="test", values=metric, sort=False)[
        ["FY2023-24", "FY2024-25", "Pooled"]
    ]

prev = results.groupby("test").prevalence_test.first()
print("test prevalence:", prev.round(3).to_dict())
print("\n=== PR-AUC")
print(wide("PR-AUC").round(3).to_string())

In [ ]:
print("=== share of bottom-10 found in the top 20%")
print(wide("found@20%").round(3).to_string())
print("\n=== ROC-AUC")
print(wide("ROC-AUC").round(3).to_string())

## Properties with no inspection history

In [ ]:
# How the best configuration does on units with no earlier inspection, where the
# last-score rule has nothing to go on.
feats = CONFIGS["+ static supplemental (everything)"]
y = units.target.astype(int).to_numpy()
pred = {m: np.full(len(units), np.nan) for m in h.MODELS}
for train_fys, test_fy in t.FOLDS:
    tr = np.flatnonzero(units.fiscal_year.isin(train_fys).to_numpy())
    te = np.flatnonzero((units.fiscal_year == test_fy).to_numpy())
    for m in h.MODELS:
        pred[m][te] = h.fit(m, units[feats].iloc[tr], y[tr]).predict_proba(units[feats].iloc[te])[:, 1]
test = units.fiscal_year.isin([fy for _, fy in t.FOLDS]).to_numpy()
rows = []
for label, mask in [("no history", test & (units.has_history == 0).to_numpy()),
                    ("with history", test & (units.has_history == 1).to_numpy())]:
    yt = y[mask]
    rows.append({"subset": label, "n": mask.sum(), "prevalence": yt.mean(), "ranker": "Random", "PR-AUC": yt.mean()})
    for m in h.MODELS:
        rows.append({"subset": label, "n": mask.sum(), "prevalence": yt.mean(), "ranker": m,
                     "PR-AUC": h.scores(yt, pred[m][mask])["PR-AUC"]})
    if label == "with history":
        rows.append({"subset": label, "n": mask.sum(), "prevalence": yt.mean(), "ranker": "Last score (lower first)",
                     "PR-AUC": h.scores(yt, BASELINES["Last score (lower first)"].to_numpy()[mask])["PR-AUC"]})
print(pd.DataFrame(rows).to_string(index=False))

## Leak check

In [ ]:
rng = np.random.default_rng(h.SEED)
shuffled = units.assign(target=rng.permutation(units.target.to_numpy()))
chk = t.run(shuffled, CONFIGS["+ static supplemental (everything)"], label="shuffled")
print(chk[chk.test == "Pooled"][["ranker", "prevalence_test", "PR-AUC"]].to_string(index=False))

## What carries the model (train through FY2023-24, test FY2024-25)

In [ ]:
feats = CONFIGS["+ static supplemental (everything)"]
train_fys, test_fy = t.FOLDS[1]
tr = units.fiscal_year.isin(train_fys).to_numpy()
te = (units.fiscal_year == test_fy).to_numpy()
best = (results[(results.test == "Pooled") & (results.label == "+ static supplemental (everything)")
                & results.ranker.isin(h.MODELS)].set_index("ranker")["PR-AUC"].idxmax())
est = h.fit(best, units.loc[tr, feats], y[tr])
r = permutation_importance(est, units.loc[te, feats], y[te], scoring="average_precision",
                           n_repeats=20, random_state=h.SEED, n_jobs=h.N_JOBS)
print("model:", best)
print(pd.Series(r.importances_mean, index=feats).sort_values(ascending=False).head(15).round(4).to_string())